In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import networkx as nx
import numpy as np
from scipy.integrate import solve_ivp
from sympy import symbols, sympify
import torch

from mlmmint.Models.MechanisticModel import ComplexODEMechanisticModel
from mlmmint.RegulatoryNetwork import RegulatoryNetwork

In [2]:
# 1) Define parameter dictionary
param_dict = {
    "pSPA": 1.0, "dSPA": 0.1, "assCOP1SPA1": 0.2, "dissCOP1SPA1": 0.05,
    "pCOP1": 1.0, "dCOP1": 0.1, "iCOP1_light": 0.1, "iCOP1_drought": 0.1,
    "iCOP1_heat_shoot": 1.0, "pCOP1_heat_shoot": 1.0, "iCOP1_SPA1_auto": 0.1,
    # Add any other missing parameters here
}

# 2) Define ODEs as lambdas that take the full state‐dict
ODE_dict = {
    "spa1_shoot": lambda s: (
        param_dict["pSPA"]
        - param_dict["dSPA"] * s["spa1_shoot"]
        - param_dict["assCOP1SPA1"] * s["cop1_shoot"] * s["spa1_shoot"]
        + param_dict["dissCOP1SPA1"] * s["cop1_spa1_shoot"]
    ),
    "cop1_shoot": lambda s: (
        param_dict["pCOP1"] * param_dict["pCOP1_heat_shoot"]**s["heat"]
        - param_dict["dCOP1"] * s["cop1_shoot"]
        - (param_dict["iCOP1_light"]*s["light"] + param_dict["iCOP1_drought"]*s["drought"])
          * param_dict["iCOP1_heat_shoot"]**s["heat"] * s["cop1_shoot"]
        - param_dict["assCOP1SPA1"] * s["cop1_shoot"] * s["spa1_shoot"]
        + param_dict["dissCOP1SPA1"] * s["cop1_spa1_shoot"]
    ),
    "cop1_spa1_shoot": lambda s: (
        param_dict["assCOP1SPA1"] * s["cop1_shoot"] * s["spa1_shoot"]
        - param_dict["dissCOP1SPA1"] * s["cop1_spa1_shoot"]
        - param_dict["iCOP1_SPA1_auto"] * s["cop1_spa1_shoot"]**2
    ),
    # … define hy5_shoot, hy5p_shoot, spa1_root, … etc. …
}

input_dict = {
    "light": lambda t: (t % 24 <= 16),  # light on for 16 hours
    "heat": lambda t: 0,                 # no heat stress
    "drought": lambda t: 0                # no drought stress
}



# 3) Get a list of the dependencies of each equation
# dependencies = {
#     "spa1_shoot":   ["cop1_shoot", "cop1_spa1_shoot"],
#     "cop1_shoot":   ["heat", "light", "drought", "spa1_shoot", "cop1_spa1_shoot"],
#     "cop1_spa1_shoot": ["cop1_shoot", "spa1_shoot"],
# }

def build_network_from_ODE_dict(ODE_dict: dict, input_dict: dict):
    dependencies = {}
    gene_names = list(ODE_dict.keys())
    input_names = list(input_dict.keys()) if input_dict is not None else ["light", "heat", "drought"]
    for var, ODE in ODE_dict.items():
        symbolic_vars = symbols(gene_names + input_names)
        symbolic_expr = sympify(
            ODE({v: symbolic_vars[i] for i, v in enumerate(gene_names)} |
                {name: symbolic_vars[len(gene_names) + j] for j, name in enumerate(input_names)})
        )
        dependencies[var] = [
            str(sym) for sym in symbolic_expr.free_symbols
            if str(sym) in gene_names or str(sym) in input_names
        ]

    G = nx.DiGraph()
    for var, ODE in ODE_dict.items():
        G.add_node(var, ODE=ODE, type="network")
    
    for input_node, time_series_func in input_dict.items():
        G.add_node(input_node, time_series_func=time_series_func, type="input")
    
    for var, deps in dependencies.items():
        for d in deps:
            G.add_edge(d, var)
    return G

G = build_network_from_ODE_dict(ODE_dict, input_dict)



# # 5) Write a wrapper to produce the derivative‐vector for scipy
# def network_ODE(t, y):
#     state = dict(zip(ODE_dict.keys(), y))
#     # Update state with input values at time t
#     for input_name, input_func in input_dict.items():
#         state[input_name] = input_func(t)
#     # Compute derivatives for each ODE variable
#     dydt = np.zeros(len(ODE_dict))
#     for i, (var, ode_func) in enumerate(ODE_dict.items()):
#         dydt[i] = ode_func(state)
#     return dydt
# 
# 
# # 6) Integrate the ODEs
# gene_names = list(ODE_dict.keys())
# y0 = np.zeros(len(gene_names)) # initial conditions
# solution = solve_ivp(network_ODE, [0, 7*24], y0, method="LSODA")


# create mechanistic model
# MM = ComplexMechanisticModel(RegulatoryNetwork(G), param_dict)
MM = ComplexODEMechanisticModel(ODE_dict, input_dict, param_dict)
gene_names = list(ODE_dict.keys())

# integrate the ODEs
t_span = (0, 7*24)  # time span for integration
t_points = torch.linspace(t_span[0], t_span[1], 1000)  # time points for integration
# Ensure time points are properly spaced to avoid numerical issues
# Ensure initial conditions are non-negative and consistent with the equations
y0 = torch.zeros(len(gene_names))  # initial conditions (non-zero positive values)
# Add error handling to catch potential numerical issues during integration
# try:
#     solution = MM.integrate(y0, t_points)
# except Exception as e:
#     print(f"Integration failed: {e}")
#     solution = None
# print(MM.regulatory_network.nodes, G.nodes)
solution = MM.integrate(y0, t_points)
solution = solve_ivp(MM.forward, t_span=[0, 7*24], y0=y0, method="LSODA")

capi_return is NULL
Call-back cb_f_in_lsoda__user__routines failed.


TypeError: zeros_like(): argument 'input' (position 1) must be Tensor, not numpy.ndarray

In [ ]:
# 7) Plot the results
fig, ax = plt.subplots(figsize=(10, 6))
for i, var in enumerate(gene_names):
    ax.plot(solution.t, solution.y[i], label=var)
ax.set_xlabel("Time (hours)")
ax.set_ylabel("Concentration")
ax.legend()
plt.title("Network Dynamics Over Time")
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
nx.draw(G, with_labels=True, node_size=2000, node_color="lightblue", font_size=10, font_weight="bold", edge_color="gray")
plt.title("Network Graph")
plt.show()

In [ ]:
adj_matrix = nx.adjacency_matrix(G).todense()
print("Adjacency Matrix:")
print(adj_matrix)

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

# 1) Helper to add const, var, and op nodes
def add_const(G, name, value):
    G.add_node(name, type='const', value=value)

def add_var(G, name):
    G.add_node(name, type='var')

def add_op(G, name, op_type, inputs):
    G.add_node(name, type='op', op_type=op_type)
    for inp in inputs:
        G.add_edge(inp, name)
    return name

# 2) Build the graph
G = nx.DiGraph()

# 2a) Add all parameter nodes (example subset; add the rest similarly)
for pname, pval in {
    "pSPA":1.0, "dSPA":0.1, "assCOP1SPA":0.2, "dissCOP1SPA":0.05,
    "pCOP1":1.2, "pCOP1_heat_shoot":1.1, "dCOP1":0.1, "dCOP1SPA":0.07,
    "iCOP1_light":0.3, "iCOP1_drought":0.4, "iCOP1_heat_shoot":1.05,
    "pHY5_shoot":0.8, "pHY5_light_shoot":0.5, "pHY5_farred_shoot":0.2,
    "iHY5_SPA":0.6, "iHY5_SPA_heat":1.02, "iHY5_SPA_hr":0.3,
    "aHY5":0.4, "dHY5":0.05, "dHY5_COP1":0.02,
    "shootHY5COPDegFactor":1.0, "transportFrac":0.1,
    "ratio_degradation_HY5_HY5p":0.5,
    # ... add root params analogously ...
}.items():
    add_const(G, pname, pval)

# Special constant for negation
add_const(G, "neg1", -1.0)

# 2b) Add variable nodes
for v in [
    "c_spa1_shoot","c_cop1_shoot","c_cop1_spa_shoot",
    "c_hy5_shoot","c_hy5p_shoot",
    "c_spa1_root","c_cop1_root","c_cop1_spa_root",
    "c_hy5_root","c_hy5p_root"
]:
    add_var(G, v)

# 2c) Build the spa1_shoot derivative subgraph
#   M2 = dSPA * c_spa1_shoot
add_op(G, "M2_spa1_shoot", "mul", ["dSPA","c_spa1_shoot"])
#   T2 = -1 * M2
add_op(G, "T2_spa1_shoot", "mul", ["neg1","M2_spa1_shoot"])
#   M3a = assCOP1SPA * c_cop1_shoot
add_op(G, "M3a_spa1_shoot","mul",["assCOP1SPA","c_cop1_shoot"])
#   M3  = M3a * c_spa1_shoot
add_op(G, "M3_spa1_shoot","mul",["M3a_spa1_shoot","c_spa1_shoot"])
#   T3  = -1 * M3
add_op(G, "T3_spa1_shoot","mul",["neg1","M3_spa1_shoot"])
#   T4  = dissCOP1SPA * c_cop1_spa_shoot
add_op(G, "T4_spa1_shoot","mul",["dissCOP1SPA","c_cop1_spa_shoot"])
#   A_spa1 = pSPA + T2 + T3 + T4  ← dc_spa1_shoot
add_op(G, "A_spa1_shoot","add",["pSPA","T2_spa1_shoot","T3_spa1_shoot","T4_spa1_shoot"])
G.nodes["A_spa1_shoot"]["deriv_of"] = "c_spa1_shoot"

# 2d) Build cop1_shoot derivative subgraph (same pattern)
add_op(G, "M1_cop1_shoot","pow",["pCOP1_heat_shoot","heat"])          # pCOP1_heat_shoot**heat
add_op(G, "T1_cop1_shoot","mul",["pCOP1","M1_cop1_shoot"])            # pCOP1 * (…)
add_op(G, "M2_cop1_shoot","mul",["dCOP1","c_cop1_shoot"])            # dCOP1 * c_cop1_shoot
add_op(G, "M3a_cop1_shoot","add",["iCOP1_light","iCOP1_drought"])    # iCOP1_light + iCOP1_drought
add_op(G, "M3b_cop1_shoot","mul",["M3a_cop1_shoot","light","drought"])# (…) * light * drought
add_op(G, "M3c_cop1_shoot","mul",["M3b_cop1_shoot","iCOP1_heat_shoot","heat"]) # * iCOP1_heat_shoot**heat
add_op(G, "M3_cop1_shoot","mul",["M3c_cop1_shoot","c_cop1_shoot"])
add_op(G, "M4a_cop1_shoot","mul",["assCOP1SPA","c_cop1_shoot"])
add_op(G, "M4_cop1_shoot","mul",["M4a_cop1_shoot","c_spa1_shoot"])
add_op(G, "T4_cop1_shoot","mul",["neg1","M4_cop1_shoot"])
add_op(G, "T5_cop1_shoot","mul",["dissCOP1SPA","c_cop1_spa_shoot"])
add_op(G, "A_cop1_shoot","add",[
    "T1_cop1_shoot", 
    "neg1"*1,  # negative sign for M2
    "M2_cop1_shoot", 
    "neg1"*1,  # negative sign for M3
    "M3_cop1_shoot", 
    "T4_cop1_shoot", 
    "T5_cop1_shoot",
])
# … mark deriv_of …
G.nodes["A_cop1_shoot"]["deriv_of"] = "c_cop1_shoot"

# (Repeat the same deconstruction for c_cop1_spa_shoot, c_hy5_shoot, c_hy5p_shoot,
#  and then for all the root variables.)

# after building the entire graph, but before drawing:
for n, data in G.nodes(data=True):
    if "type" not in data:
        data["type"] = "input"   # or "var"/"const", whichever you prefer

# # extend your colors dict accordingly:
# colors["input"] = "lightgray"


# 3) Visualize the full graph
plt.figure(figsize=(10,10))
pos = nx.spring_layout(G, k=0.8, iterations=50, scale=10)  # positions for all nodes
# color by node type
colors = {
    'const':'gold', 'var':'skyblue', 'op':'lightgreen', 'input':'lightgray',
}
nx.draw_networkx_nodes(
    G, pos,
    node_size=500,
    node_color=[colors[G.nodes[n]['type']] for n in G.nodes()]
)
nx.draw_networkx_edges(G, pos, arrowstyle='-|>', arrowsize=8)
nx.draw_networkx_labels(G, pos, font_size=8)
plt.title("Computational Graph of Shoot + Root ODE System")
plt.axis('off')
plt.show()

In [ ]:
import networkx as nx
import numpy as np
from scipy.integrate import solve_ivp

# ── 1) Prepare your graph G as before ────────────────────────────────
# G is the DiGraph with nodes of type 'const', 'input', 'var', 'op',
# and 'op' nodes have an attribute 'op_type' in {'add','mul','pow','div'},
# and certain 'add' nodes carry the attribute 'deriv_of' = <var_name>.

# ── 2) Choose an ordering of your state variables ────────────────────
vars_order = [
    "c_spa1_shoot", "c_cop1_shoot", "c_cop1_spa_shoot",
    "c_hy5_shoot",   "c_hy5p_shoot",
    "c_spa1_root",   "c_cop1_root",   "c_cop1_spa_root",
    "c_hy5_root",    "c_hy5p_root"
]

# ── 3) Pre‐compute a topological sort of op‐nodes ─────────────────────
# We only need to evaluate op‐nodes in topo order; const, input, var are seeds.
op_nodes = [n for n, d in G.nodes(data=True) if d['type']=='op']
# eval_order = [n for n in nx.topological_sort(G) if n in op_nodes]

op_graph = G.subgraph(op_nodes).copy()
# Topo–sort _only_ that subgraph:
try:
    eval_order = list(nx.topological_sort(op_graph))
except nx.NetworkXUnfeasible:
    # if you still get a cycle, this will show you which ops are involved
    print("Cycle detected in operator subgraph:", nx.find_cycle(op_graph))
    raise

light_time_series = lambda t: 1.0 if (t % 24 <= 16) else 0.0
heat_time_series = lambda t: 0.0
drought_time_series = lambda t: 0.0

# ── 4) Define your RHS function by walking the graph ────────────────
def network_rhs(t, y):
    # a) build a dict of node → value
    vals = {}
    # fill constants
    for n, d in G.nodes(data=True):
        if d['type']=='const':
            vals[n] = d['value']
    # fill inputs (time‐varying)
    vals['light']   = light_time_series(t)
    vals['heat']    = heat_time_series(t)
    vals['drought'] = drought_time_series(t)
    # fill current variables
    for var_name, y_val in zip(vars_order, y):
        vals[var_name] = y_val

    # b) compute all op‐nodes
    for node in eval_order:
        typ = G.nodes[node]['op_type']
        ins = list(G.predecessors(node))
        args = [vals[i] for i in ins]
        if typ == 'add':
            vals[node] = sum(args)
        elif typ == 'mul':
            prod = 1.0
            for a in args:
                prod *= a
            vals[node] = prod
        elif typ == 'pow':
            vals[node] = args[0] ** args[1]
        elif typ == 'div':
            vals[node] = args[0] / args[1]
        else:
            raise ValueError(f"Unknown op {typ}")

    # c) collect the derivatives
    dydt = [0.0]*len(vars_order)
    for node, d in G.nodes(data=True):
        if 'deriv_of' in d:
            var = d['deriv_of']
            idx = vars_order.index(var)
            dydt[idx] = vals[node]

    return np.array(dydt)


# ── 5) Integrate with SciPy ─────────────────────────────────────────
# initial conditions must match vars_order
# y0 = np.array([  # fill in with your measured or assumed starts
#     1.0, 0.5, 0.2, 0.3, 0.1,   # shoot
#     0.8, 0.4, 0.1, 0.2, 0.05   # root
# ])
y0 = np.array([1.0] * len(vars_order))  # initial conditions (non-zero positive values)

t_span = (0.0, 72.0)            # simulate 0 – 72 hours
t_eval = np.linspace(*t_span, 300)

sol = solve_ivp(network_rhs, t_span, y0, t_eval=t_eval, method='RK45')

# sol.y is an array (n_states × n_times); sol.t are the time points

In [ ]:
# ── 6) Plot the results ─────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")
plt.figure(figsize=(12, 6))
for i, var in enumerate(vars_order):
    plt.plot(sol.t, sol.y[i], label=var)
plt.xlabel("Time (hours)")
plt.ylabel("Concentration")
plt.title("Network Dynamics Over Time")
plt.legend()
plt.show()